# Analiza streszczeń literatury młodzieżowej


## Import bibliotek

In [ ]:
## Import bibliotek

import json
import nltk
import morfeusz2
import pandas as pd
import string
import operator as op
import itertools as it
from nltk.tokenize import word_tokenize
from nltk.corpus import PlaintextCorpusReader
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer


In [ ]:
import os
corpus_dir = os.path.abspath("./Literature")
nltk.data.path.append(corpus_dir)  # Zezwolenie NLTK na czytanie plików z tego katalogu
# punkt_tab jest już pobrany do ~/nltk_data (nltk.download('punkt_tab'))

## Utwórz korpus dokumentów

In [ ]:
corpus = PlaintextCorpusReader(corpus_dir, r'.*\.txt')
file_names = corpus.fileids()

## Wstępne przygotowanie dokumentów


In [ ]:
documents_dict = {}
for file_name in file_names:
    documents_dict[file_name] = corpus.raw(file_name)


In [ ]:
lengths = {}
for file_name in documents_dict:
    lengths[file_name] = {
        "pre": len(word_tokenize(documents_dict[file_name])),
    }
   

### Wczytanie stopwords

In [ ]:
stopwords = []
with open("stopwords_pl.txt", "r", encoding="utf-8") as stopwords_file:
    stopwords = stopwords_file.read().splitlines()
    stopwords = stopwords[4:] # Pomijamy pierwsze 4 linie, które zawierają nagłówki
# print("Stopwords:", stopwords)
    

### Lematyzacja

In [ ]:
morfer = morfeusz2.Morfeusz()  # tworzymy raz, nie przy każdym wywołaniu
# print(*morfer.analyse("Ala ma kota."), sep="\n")  # Analiza morfologiczna 
def lemmatize(text):
    segments = it.groupby(morfer.analyse(text), key=op.itemgetter(0, 1))

    def disambiguate(segment):
        # analysis = (forma, lemat, tag, ...) — wybieramy interpretację z najkrótszym tagiem
        pairs = (
            (len(analysis[2]), analysis[1])
            for _, _, analysis in segment
        )
        _, lemma = min(pairs, key=op.itemgetter(0))
        return lemma.split(":")[0].lower() if lemma else ""

    lemmas = (disambiguate(segment) for _, segment in segments)
    return " ".join(filter(str.isalpha, lemmas))

### Lematyzacja korpusu

In [ ]:
for file_name in file_names:
    documents_dict[file_name] = documents_dict[file_name].lower()
    documents_dict[file_name] = "".join(
        [char for char in documents_dict[file_name] if char not in string.punctuation]
    )
    documents_dict[file_name] = lemmatize(documents_dict[file_name])
    documents_dict[file_name] = " ".join(
        [
            word
            for word in word_tokenize(documents_dict[file_name], language="polish")
            if word not in stopwords
        ]
    )

In [ ]:
for file_name in documents_dict:
    lengths[file_name]["post"] = len(word_tokenize(documents_dict[file_name]))

In [ ]:
lengths = pd.DataFrame.from_dict(lengths, orient="index")
lengths["change"] = lengths["post"] - lengths["pre"]
lengths["change_percent"] = (lengths["change"] / lengths["pre"]) * 100
print(lengths)

In [ ]:
documents_df = pd.DataFrame.from_dict(
    documents_dict, orient="index", columns=["content"]
)
print(documents_df)

In [ ]:
matrix_tf = CountVectorizer().fit_transform(documents_df["content"])
matrix_tfidf = TfidfVectorizer().fit_transform(documents_df["content"])


In [ ]:
sparsity_tf = (1 - (matrix_tf.nnz / (matrix_tf.shape[0] * matrix_tf.shape[1]))) * 100
sparsity_tfidf = (
    1 - (matrix_tfidf.nnz / (matrix_tfidf.shape[0] * matrix_tfidf.shape[1]))
) * 100